## Figure 5A

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import ScalarFormatter


df = pd.read_csv("./result/Section5/ACMG.actionable.genes.GRCh38.vcf", sep='\t')
df.drop_duplicates(subset='Variant38', inplace=True)
values = df['pred_prob'].dropna().values

# Define score intervals
low_vals = values[values <= 0.1]
mid_vals = values[(values > 0.1) & (values < 0.9)]
high_vals = values[values >= 0.9]

# Count observations
n_low = len(low_vals)
n_mid = len(mid_vals)
n_high = len(high_vals)

n_low_str = f"{n_low:,}"
n_mid_str = f"{n_mid:,}"
n_high_str = f"{n_high:,}"

# Plot
fig, ax = plt.subplots(figsize=(6.2, 5))
bins = np.linspace(0, 1, 50)
ax.hist(mid_vals, bins=bins, color="#ebe3dd",
        label=f'Indeterminate TranSyn score (n={n_mid_str})')
ax.hist(low_vals, bins=bins, color='#47a2bb',
        label=f'Low TranSyn score (n={n_low_str})')
ax.hist(high_vals, bins=bins, color='#e24636',
        label=f'High TranSyn score (n={n_high_str})')

# Threshold line
ax.axvline(0.1, linestyle='--', color='gray', linewidth=1)
ax.axvline(0.9, linestyle='--', color='gray', linewidth=1)

# Scientific notation
formatter = ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((0, 0))  # Force scientific notation
ax.yaxis.set_major_formatter(formatter)

ax.ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
ax.yaxis.get_offset_text().set_fontsize(12)

# Refine the plot style
ax.set_xlabel('TranSyn score', fontsize=13)
ax.set_ylabel('Count', fontsize=13)
plt.tick_params(axis='both', labelsize=12)
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color('#333')

from matplotlib.lines import Line2D
markersize=12
legend_elements = [
    Line2D([0], [0],
           marker='o',
           color='none',
           label=f'Intermediate TranSyn score (n={n_mid_str})',
           markerfacecolor='#ebe3dd',
           markeredgecolor='none',
           markersize=markersize),

    Line2D([0], [0],
           marker='o',
           color='none',
           label=f'Low TranSyn score (n={n_low_str})',
           markerfacecolor='#47a2bb',
           markeredgecolor='none',
           markersize=markersize),

    Line2D([0], [0],
           marker='o',
           color='none',
           label=f'High TranSyn score (n={n_high_str})',
           markerfacecolor='#e24636',
           markeredgecolor='none',
           markersize=markersize),
]
ax.legend(handles=legend_elements, frameon=False, fontsize=11.5, loc=[0.15,0.78])

plt.tight_layout()
plt.show()

# Figure 5B

Pathogenicity of synonymous variants across genes

1. This analysis evaluates the genes in which synonymous variants have a significantly higher probability of pathogenicity than the background.

In [ ]:
import pandas as pd
df = pd.read_csv("./result/Section5/ACMG.actionable.genes.GRCh38.vcf", sep='\t')
print(len(df))

### A) Statistical analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import binomtest
from statsmodels.stats.multitest import multipletests

def classify(p):
	"""Map scores to categories."""
	if p >= 0.9:
		return 'pathogenic'
	elif p <= 0.1:
		return 'benign'
	else:
		return 'indeterminate'
df = df.copy()
df['label'] = df['pred_prob'].apply(classify) # Assign categories

# Gene-level statistics
summary = (
	df
	.groupby(['gene', 'label'])
	.size()
	.unstack(fill_value=0)
)
for col in ['pathogenic', 'benign', 'indeterminate']:
	if col not in summary.columns:
		summary[col] = 0
summary['total'] = summary.sum(axis=1)

# Filter by sample size (at least 20 samples)
min_n = 20
summary = summary[summary['total'] >= min_n].copy()

# Calculate global proportions
global_rates = df['label'].value_counts(normalize=True)
p_path = global_rates.get('pathogenic', 0)
p_benign = global_rates.get('benign', 0)
p_indet = global_rates.get('indeterminate', 0)

# Binomial tests
def binom_pval(k, n, p0, alternative):
	if n == 0:
		return np.nan
	k = int(k)
	n = int(n)
	return binomtest(k, n, p0, alternative=alternative).pvalue
summary['p_path'] = summary.apply(
	lambda r: binom_pval(r['pathogenic'], r['total'], p_path, 'greater'), axis=1
)
summary['p_benign'] = summary.apply(
	lambda r: binom_pval(r['benign'], r['total'], p_benign, 'greater'), axis=1
)
summary['p_indet'] = summary.apply(
	lambda r: binom_pval(r['indeterminate'], r['total'], p_indet, 'greater'), axis=1
)

# Multiple-testing correction
for col in ['p_path', 'p_benign', 'p_indet']:
	mask = summary[col].notna()
	summary.loc[mask, col + '_adj'] = multipletests(
		summary.loc[mask, col],
		method='fdr_bh'
	)[1]

# Proportion
summary['path_ratio'] = summary['pathogenic'] / summary['total']
summary['benign_ratio'] = summary['benign'] / summary['total']
summary['indet_ratio'] = summary['indeterminate'] / summary['total']

### B) Plot


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D

# ---- 1. Category -> color ----
category_colors = {
    'Cancer':         "#fd8804",
    'Cardiovascular': "#155f78",
    'Miscellaneous':  '#dbdee3',
}
default_color = '#000000'   

# ACMG gene information
acmg_actions = pd.read_csv(
    "./result/Section5/ACMG_actions.csv",
    usecols=['Gene', 'Phenotype Category']
)

top_n = 20
df_plot = (
    summary[summary['p_path_adj'] < 0.05]
    .sort_values('path_ratio', ascending=False)
    .head(top_n)
    .reset_index()
    .merge(acmg_actions, left_on='gene', right_on='Gene', how='left')
    .drop('Gene', axis=1)
)
df_plot['Phenotype Category'] = df_plot['Phenotype Category'].fillna('Unknown')
df_plot = df_plot.set_index('gene')

# ---- 2. Derive colors from the same dictionary ----
bar_colors = df_plot['Phenotype Category'].map(category_colors).fillna(default_color)

# ---- 3. Plot ----
plt.figure(figsize=(4, 5))
plt.barh(
    df_plot.index[::-1],
    df_plot['path_ratio'][::-1],
    color=bar_colors[::-1],   # Reverse consistently with the y-axis order
    height=0.65
)

plt.xlabel('High TranSyn score sSNV ratio', fontsize=13, x=0.45)
plt.yticks(fontstyle='italic')

# ---- 4. Generate the legend from the same dictionary ----
markersize = 12
legend_elements = [
    Line2D([0], [0],
           marker='o',
           color='none',
           label=label,
           markerfacecolor=color,
           markeredgecolor='none',
           markersize=markersize)
    for label, color in category_colors.items()
]
plt.legend(handles=legend_elements,
           frameon=False,
           fontsize=11,
           loc='lower right',
           handletextpad=0.2,
           bbox_to_anchor=[1.03, 0.02])

# Spines
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_linewidth(.5)
    spine.set_color('#333')
plt.tick_params(axis='both', labelsize=11)

plt.tight_layout()
plt.show()

## Figure 5C,D

1. Here, miRNA binding is represented as a Boolean value indicating whether a variant is located within a miRNA-binding site. Fisher's exact test is used for this binary feature.
2. However, in the overall dataset, benign sSNVs are more frequently located within miRNA-binding sites, contrary to the original hypothesis.

### A) Overall statistical analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact, chi2_contingency

# 1. Filter the data
df_mi = df[df['label'].isin(['benign', 'pathogenic'])].copy()

# 2. Construct a binary variable
df_mi['binding'] = df_mi['miRNA binding'].notna().astype(int)

# 3. Construct the contingency table
table = pd.crosstab(df_mi['label'], df_mi['binding'])

# Ensure a consistent column order
table = table.reindex(index=['benign', 'pathogenic'], columns=[0, 1], fill_value=0)
print(table)

# 4. Fisher exact test
oddsratio, p_value = fisher_exact(table.values)

print("Fisher p-value:", p_value)
print("Odds ratio:", oddsratio)

# 5. Chi-square test
chi2, p_chi2, _, _ = chi2_contingency(table.values)
print("Chi-square p-value:", p_chi2)

### B) Overall plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import fisher_exact

# Calculate counts, proportions, and intervals from the observed data instead of using hard-coded values from the original figure.
comparison_table = pd.crosstab(df_mi["label"], df_mi["binding"]).reindex(
    index=["benign", "pathogenic"], columns=[0, 1], fill_value=0,
)
group_totals = comparison_table.sum(axis=1).to_numpy()
binding_counts = comparison_table[1].to_numpy()
binding_proportions = binding_counts / group_totals
confidence_half_widths = 1.96 * np.sqrt(
    binding_proportions * (1 - binding_proportions) / group_totals
)
odds_ratio, fisher_probability = fisher_exact(comparison_table.to_numpy())
significance = "***" if fisher_probability < 0.001 else "**" if fisher_probability < 0.01 else "*" if fisher_probability < 0.05 else "ns"
figure, axis = plt.subplots(figsize=(4, 5))
axis.bar(
    [0, 1], binding_proportions, yerr=confidence_half_widths,
    capsize=4, width=0.4, color=["#47a2bb", "#e24636"],
)
axis.set_xticks([0, 1])
axis.set_xticklabels(["Low TranSyn\nscore", "High TranSyn\nscore"], rotation=0)
axis.set_ylabel("miRNA-binding proportion", fontsize=12)
axis.tick_params(axis="both", labelsize=11, labelrotation=0)
axis.set_ylim(0, float((binding_proportions + confidence_half_widths).max()) * 1.4)
axis.text(
    0.5, 0.96, f"Fisher's exact test: {significance}\nOdds ratio: {odds_ratio:.3f}",
    transform=axis.transAxes, ha="center", va="top", fontsize=11,
)
for group_index, (proportion, half_width) in enumerate(zip(binding_proportions, confidence_half_widths)):
    axis.text(
        group_index, proportion + half_width + 0.005,
        f"{proportion:.3f}±{half_width:.3f}", ha="center", va="bottom", fontsize=10,
    )
for spine in axis.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color("#333333")
figure.tight_layout()
plt.show()


### C) Gene-level statistical analysis

In [ ]:
import pandas as pd
import numpy as np

df_gene_mi = df_mi.copy()
df_gene_mi["label_bin"] = (df_gene_mi["label"] == "pathogenic").astype(int)

def compute_or(g):
	table = pd.crosstab(g["binding"], g["label_bin"])
	
	# Ensure a 2x2 structure
	table = table.reindex(index=[0,1], columns=[0,1], fill_value=0)
	
	# Add-one smoothing to avoid zeros
	a = table.loc[1,1] + 1   # binding=1 & pathogenic
	b = table.loc[1,0] + 1   # binding=1 & benign
	c = table.loc[0,1] + 1   # binding=0 & pathogenic
	d = table.loc[0,0] + 1   # binding=0 & benign
	
	or_value = (a / b) / (c / d)
	return pd.Series({"OR": or_value, "n": len(g)})

from scipy.stats import fisher_exact

def compute_stats(g):
    table = pd.crosstab(g["binding"], g["label_bin"])
    table = table.reindex(index=[0,1], columns=[0,1], fill_value=0)

    oddsratio, p = fisher_exact(table)

    return pd.Series({
        "OR": oddsratio,
        "log2OR": np.log2(oddsratio) if oddsratio > 0 else np.nan,
        "p": p,
        "n": len(g),
        "binding_n": table.loc[1].sum()
    })

gene_stats = (
    df_gene_mi
    .groupby("gene")
    .apply(compute_stats)
    .reset_index()
)
gene_stats = gene_stats[
    gene_stats["binding_n"] > 0
]

gene_stats.sort_values("log2OR")
# gene_or = df_gene_mi.groupby("gene").apply(compute_or).reset_index()
# Apply Benjamini-Hochberg correction to all included genes without selecting the correction set by odds-ratio magnitude or plot annotations.
from statsmodels.stats.multitest import multipletests

gene_stats["fdr"] = multipletests(gene_stats["p"], method="fdr_bh")[1]
print(f"Number of miRNA gene tests included in BH correction: {len(gene_stats)}")


### D) Plot of the refined statistical analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# The correction set remains 78 genes; the plot displays only finite raw odds ratios greater than zero.
plot_table = gene_stats.replace([np.inf, -np.inf], np.nan)
plot_table = plot_table.dropna(subset=["OR"])
plot_table = plot_table[(plot_table["binding_n"] > 0) & (plot_table["OR"] > 0)].copy()
plot_table["log2OR"] = np.log2(plot_table["OR"])
annotation_table = plot_table[
    (plot_table["log2OR"].abs() > 1)
    & (plot_table["fdr"] < 0.05)
    & (plot_table["n"] >= 10)
].sort_values("fdr").head(5)

figure, axis = plt.subplots(figsize=(4.8, 5))
axis.set_xscale("log")
axis.axvspan(1, 10, alpha=0.08, color="gray")
axis.scatter(plot_table["n"], plot_table["log2OR"], alpha=0.45, color="#152e8b", linewidth=0)
axis.axhline(0, linestyle="--", linewidth=1, color="#152e8b")
axis.set_ylim(-4.5, 4.5)
axis.set_xlabel("Sample size (log scale)", fontsize=12)
axis.set_ylabel("log$_2$(Odds Ratio)", fontsize=12)
axis.tick_params(axis="both", labelsize=11, labelrotation=0)

# Place gene labels at evenly spaced positions on the right and connect them to their original points to prevent overlapping labels.
annotation_table = annotation_table.sort_values("log2OR")
for annotation_index, (_, gene_result) in enumerate(annotation_table.iterrows()):
    axis.annotate(
        gene_result["gene"],
        xy=(gene_result["n"], gene_result["log2OR"]),
        xycoords="data",
        xytext=(1.03, 0.27 + 0.10 * annotation_index),
        textcoords="axes fraction",
        ha="left", va="center", fontsize=10, fontstyle="italic",
        annotation_clip=False,
        arrowprops={"arrowstyle": "-", "color": "gray", "lw": 0.8},
    )
for spine in axis.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color("#333333")
figure.subplots_adjust(right=0.82, bottom=0.14)
plt.show()


# Figure 5E,F

Association with mRNA stability

### A) Overall statistical analysis


In [ ]:
# Use a consistent direction for Cliff's delta: High minus Low; positive values indicate larger feature values in the High group.
from scipy.stats import mannwhitneyu


df_mrna = df[df['label'].isin(['benign', 'pathogenic'])].copy()
benign_vals = df_mrna[df_mrna['label'] == 'benign']['delta_MFE'].dropna()
path_vals = df_mrna[df_mrna['label'] == 'pathogenic']['delta_MFE'].dropna()

# Statistical analysis
stat, p_value = mannwhitneyu(
	benign_vals,
	path_vals,
	alternative='two-sided'
)

print("Mann-Whitney p:", p_value)

# effect size
import numpy as np
def cliffs_delta(x, y):
	"""Calculate P(X > Y) - P(X < Y), with X representing the High group and Y the Low group."""
	x = np.array(x)
	y = np.array(y)
	return (np.sum(x[:, None] > y) - np.sum(x[:, None] < y)) / (len(x) * len(y))

delta = cliffs_delta(path_vals.values, benign_vals.values)
print("Cliff's delta:", delta)

### B) Overall plot

In [ ]:
# Use a consistent direction for Cliff's delta: High minus Low; positive values indicate larger feature values in the High group.
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from matplotlib.lines import Line2D

# The overall analysis is restricted to actionable genes covered by the input file and does not represent the entire genome.
comparison_variants = df[df["label"].isin(["benign", "pathogenic"])]
low_score_values = comparison_variants.loc[
    comparison_variants["label"] == "benign", "delta_MFE"
].dropna()
high_score_values = comparison_variants.loc[
    comparison_variants["label"] == "pathogenic", "delta_MFE"
].dropna()
mann_whitney_statistic, probability_value = mannwhitneyu(
    low_score_values, high_score_values, alternative="two-sided",
)
effect_size = 1 - 2 * mann_whitney_statistic / (len(low_score_values) * len(high_score_values))
significance = "***" if probability_value < 0.001 else "**" if probability_value < 0.01 else "*" if probability_value < 0.05 else "ns"
figure, axis = plt.subplots(figsize=(4, 5))
sns.kdeplot(
    low_score_values, label="Low TranSyn\nscore", color="#47a2bb",
    fill=True, alpha=0.3, ax=axis,
)
sns.kdeplot(
    high_score_values, label="High TranSyn\nscore", color="#e24636",
    fill=True, alpha=0.3, ax=axis,
)
axis.set_xlabel(r"$\Delta$MFE", fontsize=12)
axis.set_ylabel("Density", fontsize=12)
axis.tick_params(axis="both", labelsize=11, labelrotation=0)
# Restore the circular legend from the original figure and retain two-line labels for prediction groups.
legend_handles = [
    Line2D(
        [0], [0], marker="o", linestyle="none", color="none",
        label="High TranSyn\nscore", markerfacecolor="#e24636",
        markeredgecolor="none", markersize=12,
    ),
    Line2D(
        [0], [0], marker="o", linestyle="none", color="none",
        label="Low TranSyn\nscore", markerfacecolor="#47a2bb",
        markeredgecolor="none", markersize=12,
    ),
]
axis.legend(
    handles=legend_handles, frameon=False, fontsize=11,
    loc="upper right", handletextpad=0.2,
)
# Restore the gray translucent annotation box and position it in axes-relative coordinates without depending on variables from other plotting cells.
axis.text(
    0.5, 0.72,
    f"Mann-Whitney test: {significance}\nEffect size: {effect_size:.3f}",
    transform=axis.transAxes,
    ha="center", va="top", fontsize=11,
    bbox={"boxstyle": "round", "facecolor": "#e6e6e6", "alpha": 0.5},
)
axis.set_ylim(0, axis.get_ylim()[1] * 1.05)
for spine in axis.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color("#333333")
figure.tight_layout()
plt.show()


### C) Gene-level statistical analysis

1. For TTR and PTEN, deltaMFE is higher in the pathogenic group than in the benign group, suggesting that pathogenic sSNVs destabilize mRNA structure in these genes.

In [ ]:
# Use a consistent direction for Cliff's delta: High minus Low; positive values indicate larger feature values in the High group.
from adjustText import adjust_text

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests


def cliffs_delta(x, y):
	"""Calculate P(X > Y) - P(X < Y), with X representing the High group and Y the Low group."""
	x = np.array(x)
	y = np.array(y)
	if len(x) == 0 or len(y) == 0:
		return np.nan
	return (np.sum(x[:, None] > y) - np.sum(x[:, None] < y)) / (len(x) * len(y))

# Gene-level statistics
def calc_stats(g):
	benign = g[g['label'] == 'benign']['delta_MFE'].dropna()
	path = g[g['label'] == 'pathogenic']['delta_MFE'].dropna()
	if len(benign) < 2 or len(path) < 2:
		return pd.Series({'effect': np.nan, 'pvalue': np.nan, 'count': len(g)})
	try:
		effect = cliffs_delta(path.values, benign.values)
		_, p = mannwhitneyu(benign, path, alternative='two-sided')
		return pd.Series({'effect': effect, 'pvalue': p, 'count': len(g)})
	except:
		return pd.Series({'effect': np.nan, 'pvalue': np.nan, 'count': len(g)})


df_mrna = df[df['label'].isin(['benign', 'pathogenic'])].copy()
gene_stats = df_mrna.groupby('gene').apply(calc_stats).reset_index()
gene_stats = gene_stats.dropna()

# FDR correction
gene_stats['pvalue'] = gene_stats['pvalue'].replace(0, 1e-300)
gene_stats['p_adj'] = multipletests(gene_stats['pvalue'], method='fdr_bh')[1]
gene_stats['neg_log_p'] = -np.log10(gene_stats['p_adj'])

# Volcano plot
plt.figure(figsize=(6.5, 5))
sns.set_style("ticks")

# Significance categories
alpha = 0.05
gene_stats['sig'] = gene_stats['p_adj'] < alpha

# Draw the scatter plot
plot = sns.scatterplot(
	data=gene_stats,
	x='effect',
	y='neg_log_p',
	size='count',
	hue='effect',
	palette='RdBu_r',
	sizes=(40, 400),
	alpha=0.9,
	edgecolor='w'
)

# reference lines
plt.axvline(0, color='#333', linestyle='--', linewidth=1, alpha=0.6)
plt.axhline(-np.log10(alpha), color='#333', linestyle='--', linewidth=1, alpha=0.6)

# labeling top genes
top_genes = gene_stats.sort_values('neg_log_p', ascending=False).head(5)
texts = []
for _, r in top_genes.iterrows():
	texts.append(
		plt.text(r["effect"], r["neg_log_p"], r["gene"], fontsize=11, fontstyle='italic',)
	)
adjust_text(
	texts,
	arrowprops=dict(arrowstyle='-', color='black', lw=1),
	expand_points=(1.5, 1.5),  # Expand the influence area of points
	expand_text=(1.2, 1.2),    # Expand the influence area of text
)

# labels
plt.xlabel("Effect size", fontsize=12)
plt.ylabel(r"$-\log_{10}$(FDR)", fontsize=12)
plt.tick_params(axis='both', labelsize=12)

# Organize the legend
# TODO Change the effect legend title to Effect size and the count title to Count
new_labels = []
handles, labels = plt.gca().get_legend_handles_labels()
for label in labels:
	if label == 'effect':
		new_labels.append('Effect size')
	elif label == 'count':
		new_labels.append('Count')
	else:
		new_labels.append(label)
plt.legend(
	handles,
	new_labels,
	loc='lower right',
	frameon=False,
	ncol=2,
	fontsize=10,
	columnspacing=.3,
	borderaxespad=0.1,
	handletextpad=0.3,
)

# plt.legend(
# 	handles,
# 	labels,
# 	# bbox_to_anchor=(1, -0.2),
# 	loc='lower left',
# 	frameon=False,
# 	# labelspacing=1.1, 
# 	ncol=2,
# 	fontsize=10,
# )

# styling
ax = plt.gca()
for spine in ax.spines.values():
	spine.set_linewidth(0.5)
	spine.set_color('#333')

plt.tight_layout()
plt.show()


# Figure 5G,H | Figure S3B

Association with codon features

### A) Overall statistical analysis

In [ ]:
# Use a consistent direction for Cliff's delta: High minus Low; positive values indicate larger feature values in the High group.
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import mannwhitneyu

import numpy as np
def cliffs_delta(x, y):
	"""Calculate P(X > Y) - P(X < Y), with X representing the High group and Y the Low group."""
	x = np.array(x)
	y = np.array(y)
	return (np.sum(x[:, None] > y) - np.sum(x[:, None] < y)) / (len(x) * len(y))

# 1 Map labels
result_df = pd.read_csv("./result/Section5/ACMG.actionable.genes.GRCh38.vcf", sep='\t')
def classify(p):
	"""Map scores to categories."""
	if p >= 0.9:
		return 'pathogenic'
	elif p <= 0.1:
		return 'benign'
	else:
		return 'indeterminate'
result_df['label'] = result_df['pred_prob'].apply(classify) # Assign categories
plot_df = result_df.copy()
plot_df = plot_df[plot_df['label'].isin(['benign', 'pathogenic'])]

# Calculate statistical significance between the two groups
stats = []
for col in ["delta_gtai", "delta_fr"]:
	g1 = plot_df.loc[plot_df["label"] == "pathogenic", col].dropna()
	g0 = plot_df.loc[plot_df["label"] == "benign", col].dropna()
	u, p = mannwhitneyu(g1, g0, alternative="two-sided")
	# stats[col] = p
	delta = cliffs_delta(g1, g0)
	stats.append(
		{
			'attribute':  col,
			'p_value': p,
			'effect_size': delta
		})
stats_df = pd.DataFrame(stats)
# Apply Benjamini-Hochberg correction to the two features in this overall codon analysis.
from statsmodels.stats.multitest import multipletests

stats_df["fdr"] = multipletests(stats_df["p_value"], method="fdr_bh")[1]


### B) Overall plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Configure the plot style
sns.set_style("ticks")
plt.rcParams.update({
    "font.size": 16,
    "axes.linewidth": 1.3,
    "xtick.major.width": 1.3,
    "ytick.major.width": 1.3,
})
palette_deep = {
    "pathogenic": "#e24636",
    "benign": "#47a2bb"
}
palette_light = {
    "pathogenic": "#ff5644",
    "benign": "#94d5ec"
}

fig, axes = plt.subplots(1, 2, figsize=(10, 6), sharey=False)

# Plot
for ax, col, title in zip(
    axes,
    ["delta_gtai", "delta_fr"],
    ["ΔgtAI", "ΔCodon frequency in 1000G"]
):
    sns.violinplot(
        data=plot_df,
        x="label",
        y=col,
        palette=palette_deep,
        inner=None,                 # Do not use the built-in seaborn box
        linewidth=1.3,
        saturation=1.0,
        cut=0,
        scale="width",
        color=None,
        ax=ax
    )
    sns.boxplot(
    data=plot_df,
    x="label",
    palette=palette_light,
    y=col,
    width=0.25,                
    showcaps=False,
    showfliers=False,
    boxprops=dict(
        edgecolor="#333",
        linewidth=1.3
    ),
    medianprops=dict(color="#333", linewidth=1.3),
    whiskerprops=dict(color="#333", linewidth=1.3),
    capprops=dict(color="#333", linewidth=1.3),
    ax=ax
    )
    ax.set_xlabel("")
    ax.set_ylabel(title, fontsize=16)
    ax.tick_params(axis='both', labelsize=14)
    # Significance annotation
    p = stats_df[stats_df['attribute'] == col].iloc[0]['fdr']
    effect_size = round(stats_df[stats_df['attribute'] == col].iloc[0]['effect_size'], 3)
    label = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    # Use axes-relative positioning and display only the significance symbol to avoid overlap or clipping.
    ax.text(
        0.5, 0.96, label,
        transform=ax.transAxes,
        ha="center", va="top", fontsize=20,
    )

    # Retain the effect size and place it on a separate line from the significance symbol using axes-relative positioning to avoid text overlap.
    ax.text(
        0.5, 0.89, f"Effect size: {effect_size:.3f}",
        transform=ax.transAxes,
        ha="center", va="top", fontsize=14,
    )

    # Show all spines
    for spine in ax.spines.values():
        spine.set_linewidth(0.5)
        spine.set_color("#333")
    
    # NOTE Wrap long text onto multiple lines
    ax.set_xticklabels([
        "Low TranSyn\nscore",
        "High TranSyn\nscore"
    ], rotation=0)

plt.tight_layout()
plt.show()

### C) Gene-level statistical analysis

1. Delta = P(transition | pathogenic, gene) - P(transition | benign, gene), representing the difference between pathogenic and benign codon-transition probabilities within a given gene.

|Delta|Interpretation|
|---|---|
|>0|The transition is more enriched in pathogenic than benign variants|
|<0|The transition is more common in benign variants|
|=0|The transition has similar proportions in both classes, with no clear association|

2. heterogeneity = |delta1| + |delta2| + |delta3|

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import chi2_contingency
from statsmodels.stats.multitest import multipletests
from matplotlib.lines import Line2D

# 0. Prepare the data
df = plot_df.copy()
# Optionally filter genes with insufficient sample sizes
gene_count = df["gene"].value_counts()
valid_genes = gene_count[gene_count >= 30].index
df = df[df["gene"].isin(valid_genes)].copy()

# 1. Calculate gene-level transition distributions
gene_dist = (
	df.groupby(
		["gene", "label", "codon_preferred_transition"]
	)
	.size()
	.reset_index(name="count")
)
# Normalize within each gene and label
gene_dist["prop"] = (
	gene_dist.groupby(["gene", "label"])["count"]
	.transform(lambda x: x / x.sum())
)

# 2. Pivot and calculate delta
pivot = gene_dist.pivot_table(
	index=["gene", "codon_preferred_transition"],
	columns="label",
	values="prop",
	fill_value=0
).reset_index()
# Ensure that required columns exist
for col in ["benign", "pathogenic"]:
	if col not in pivot.columns:
		pivot[col] = 0

# delta
# pathogenic proportion - benign proportion
pivot["delta"] = (
	pivot["pathogenic"] - pivot["benign"]
)

# 3. heterogeneity score
gene_score = (
	pivot.groupby("gene")["delta"]
	.apply(lambda x: np.abs(x).sum())
	.reset_index(name="heterogeneity")
)

# 4. Statistical test (2x3 chi-square)
def gene_test(g):
	table = pd.crosstab(
		g["label"],
		g["codon_preferred_transition"]
	)
	# Require at least a 2x2 table
	if table.shape[0] < 2 or table.shape[1] < 2:
		return pd.Series({
			"pvalue": 1.0,
			"chi2": 0
		})
	chi2, p, _, _ = chi2_contingency(table)
	return pd.Series({
		"pvalue": p,
		"chi2": chi2
	})
gene_p = (
	df.groupby("gene")
	.apply(gene_test)
	.reset_index()
)

# 5. FDR correction
gene_p["fdr"] = multipletests(
	gene_p["pvalue"],
	method="fdr_bh"
)[1]

# 6. merge
gene_score = gene_score.merge(
	gene_p,
	on="gene"
)

# 7. Calculate gene sample sizes
gene_n = (
	df.groupby("gene")
	.size()
	.reset_index(name="n")
)

gene_score = gene_score.merge(
	gene_n,
	on="gene"
)

# 8. Use BH-adjusted P values on the plot y-axis
gene_score["neglog10p"] = -np.log10(
	gene_score["fdr"] + 1e-300
)

### D) Gene-level significance plot

In [ ]:
from adjustText import adjust_text
import matplotlib.pyplot as plt

plt.figure(figsize=(6.5, 4.5))

# Heterogeneity and significance
significant = gene_score["fdr"] < 0.05
non_significant = ~significant

# Plot significant points
plt.scatter(
	gene_score.loc[significant, "heterogeneity"],
	gene_score.loc[significant, "neglog10p"],
	alpha=0.5,
	color="#e24636"  # Significant-point color
)

# Plot non-significant points
plt.scatter(
	gene_score.loc[non_significant, "heterogeneity"],
	gene_score.loc[non_significant, "neglog10p"],
	alpha=0.5,
	color="#47a2bb"  # Non-significant-point color
)

# sig baseline
plt.axhline(
	-np.log10(0.05),
	linestyle="--",
	linewidth=0.8,
	color="#666"
)

# Annotate the most extreme genes
extreme = gene_score[gene_score["fdr"] < 0.05].nlargest(10, "heterogeneity")
texts = []
for _, r in extreme.iterrows():
	texts.append(
		plt.text(r["heterogeneity"] + 0.005, 
				 r["neglog10p"], 
				 r["gene"], 
				 fontsize=10, 
				 fontstyle='italic')
	)

adjust_text(
	texts,
	arrowprops=dict(arrowstyle='-', color='gray', lw=1),
	expand_text=(1.2, 1.2),
	expand_points=(1.2, 1.2)
)

# Configure the x- and y-axes    
plt.xlabel("Heterogeneity", fontsize=12)
plt.ylabel("-log10(BH FDR)", fontsize=12)
plt.tick_params(axis='both', labelsize=11)

# Set spine widths
ax = plt.gca()
for spine in ax.spines.values():
	spine.set_linewidth(.5)
	spine.set_color('#333')

plt.tight_layout()
plt.show()


### D) Detailed gene-level plot

In [ ]:
from matplotlib.lines import Line2D

# Select top genes for delta interpretation
top_gene_df = (
	gene_score
	.query("fdr < 0.05")
	.sort_values(
		"heterogeneity",
		ascending=False
	)
	.head(10)
)

top_genes = top_gene_df["gene"]

sub = pivot[
	pivot["gene"].isin(top_genes)
].copy()

# delta barplot
palette = {
	"unpreferred_to_preferred": "#47a2bb",
	"preferred_to_unpreferred": "#e24636",
	"no_change": "#e1ddd6"
}

plt.figure(figsize=(7.5, 4))
sns.barplot(
	data=sub,
	x="gene",
	y="delta",
	hue="codon_preferred_transition",
	palette=palette
)

# legend
markersize = 12
legend_elements = [
	Line2D(
		[0], [0],
		marker='o',
		linestyle='none',
		color='none',
		label='Unpreferred to preferred',
		markerfacecolor=palette[
			"unpreferred_to_preferred"
		],
		markeredgecolor='None',
		markersize=markersize
	),
	Line2D(
		[0], [0],
		marker='o',
		linestyle='none',
		color='none',
		label='Preferred to unpreferred',
		markerfacecolor=palette[
			"preferred_to_unpreferred"
		],
		markeredgecolor='None',
		markersize=markersize
	),
	Line2D(
		[0], [0],
		marker='o',
		linestyle='none',
		color='none',
		label='No change',
		markerfacecolor=palette[
			"no_change"
		],
		markeredgecolor='None',
		markersize=markersize
	)
]
plt.legend(
	handles=legend_elements,
	frameon=False,
	fontsize=11,
	ncol=3,
	loc='upper center',
	bbox_to_anchor=[0.5, 1.15],
	labelspacing=0.5,
	columnspacing=0.5,
)

# axis
plt.xlabel("")
plt.ylabel("Delta", fontsize=12)
plt.tick_params(axis='both', labelsize=12)
plt.axhline(
	0,
	linestyle="-",
	linewidth=.5,
	color="#333"
)

plt.xticks(
	rotation=0,
	fontstyle='italic'
)

# border
ax = plt.gca()
for spine in ax.spines.values():
	spine.set_linewidth(0.5)
	spine.set_color('#333')

plt.tight_layout()
plt.show()

In [ ]:
# 11. Export the result table
display(
	gene_score.sort_values(
		"heterogeneity",
		ascending=False
	).head(20)
)

# Figure 5H,I | Figure S3A,C

 Association with splicing


### A) Overall statistical analysis

In [ ]:
# Use a consistent direction for Cliff's delta: High minus Low; positive values indicate larger feature values in the High group.
import pandas as pd
import numpy as np

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

import seaborn as sns
import matplotlib.pyplot as plt

# 1. Load data
def classify(p):
	"""Map scores to categories."""
	if p >= 0.9:
		return 'pathogenic'
	elif p <= 0.1:
		return 'benign'
	else:
		return 'indeterminate'

def cliffs_delta(x, y):
	"""Calculate P(X > Y) - P(X < Y), with X representing the High group and Y the Low group."""
	x = np.array(x)
	y = np.array(y)
	if len(x) == 0 or len(y) == 0:
		return np.nan
	return (np.sum(x[:, None] > y) - np.sum(x[:, None] < y)) / (len(x) * len(y))

df = pd.read_csv(
	"./result/Section5/ACMG.actionable.genes.GRCh38.vcf",
	sep='\t'
)

df = df.copy()

df['label'] = df['pred_prob'].apply(classify)

# 2. Retain only pathogenic and benign groups
df = df[
	df['label'].isin(['pathogenic', 'benign'])
].copy()

# 3. SpliceAI features to analyze
splice_cols = [
	'SpliceAI_acc_gain',
	'SpliceAI_acc_loss',
	'SpliceAI_don_gain',
	'SpliceAI_don_loss'
]

# 4. Convert data to numeric values to avoid strings
for col in splice_cols:
	df[col] = pd.to_numeric(df[col], errors='coerce')

# 5. Mann-Whitney U test
stats_results = []
for col in splice_cols:
	benign = df.loc[
		df['label'] == 'benign',
		col
	].dropna()
	pathogenic = df.loc[
		df['label'] == 'pathogenic',
		col
	].dropna()

	n1 = len(benign)
	n2 = len(pathogenic)

	# Mann-Whitney U
	stat, p = mannwhitneyu(
		benign,
		pathogenic,
		alternative='two-sided'
	)

	# Rank-biserial correlation (effect size)
	# rbc = 1 - (2 * stat) / (n1 * n2)
	effect_size = cliffs_delta(pathogenic, benign)
	stats_results.append({
		"feature": col,
		"U_statistic": stat,
		"pvalue": p,
		# "RBC": rbc,
		"effect_size": effect_size,
		"benign_mean": benign.mean(),
		"pathogenic_mean": pathogenic.mean(),
		"benign_median": benign.median(),
		"pathogenic_median": pathogenic.median(),
		"n_benign": n1,
		"n_pathogenic": n2,
	})

stats_df = pd.DataFrame(stats_results)
# 6. FDR correction
stats_df["fdr"] = multipletests(
	stats_df["pvalue"],
	method="fdr_bh"
)[1]

# 7. Export statistical results
print("\n=== Mann-Whitney U test ===")
display(
	stats_df.sort_values("pvalue")
)

### B) Overall plot

| RBC   | effect |
| ------- | ------ |
| < 0.1   | Very weak     |
| 0.1~0.3 | Weak      |
| 0.3~0.5 | Moderate     |
| >0.5    | Strong      |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# configs
palette = {
    "benign": "#47a2bb",
    "pathogenic": "#e24636"
}
mapping = {
    'Acceptor gain': 'SpliceAI_acc_gain',
    'Acceptor loss': 'SpliceAI_acc_loss',
    'Donor gain': 'SpliceAI_don_gain',
    'Donor loss': 'SpliceAI_don_loss'
}

for k, v in mapping.items():
    # plt.figure(figsize=(5, 4.5))
    plt.figure(figsize=(3, 2.8))
    sns.kdeplot(
        data=df,
        x=v,
        hue="label",
        fill=True,
        common_norm=False,
        alpha=0.35,
        linewidth=1.5,
        palette=palette
    )
    # Retain BH-adjusted values for the four overall tests and the original Cliff's delta values; continue to hide the legend.
    feature_statistics = stats_df.loc[stats_df["feature"] == v].iloc[0]
    adjusted_probability = feature_statistics["fdr"]
    # When floating-point underflow produces zero, do not present it as an exact zero probability.
    probability_label = (
        "< 1e-300" if adjusted_probability == 0
        else f"= {adjusted_probability:.2e}"
    )
    plt.title(
        f"BH FDR {probability_label}\n"
        f"Effect size: {feature_statistics['effect_size']:.3f}",
        fontsize=11,
    )

    # axis
    plt.xlabel(k, fontsize=12)
    plt.ylabel("Density", fontsize=12)
    plt.tick_params(
        axis='both',
        labelsize=11
    )

    # The combined figure defines the groups globally, so individual overall SpliceAI panels do not display legends.
    legend = plt.gca().get_legend()
    if legend is not None:
        legend.remove()

    # border
    ax = plt.gca()
    for spine in ax.spines.values():
        spine.set_visible(False)
        # spine.set_color('#333')
        # ax.spines['top'].set_visible(False)
        # ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()

### C) Gene-level statistical analysis

In [ ]:
# Use a consistent direction for Cliff's delta: High minus Low; positive values indicate larger feature values in the High group.
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt


def cliffs_delta(x, y):
	"""Calculate P(X > Y) - P(X < Y), with X representing the High group and Y the Low group."""
	x = np.array(x)
	y = np.array(y)
	if len(x) == 0 or len(y) == 0:
		return np.nan
	return (np.sum(x[:, None] > y) - np.sum(x[:, None] < y)) / (len(x) * len(y))

# 1. Load data
def classify(p):
	if p >= 0.9:
		return 'pathogenic'
	elif p <= 0.1:
		return 'benign'
	else:
		return 'indeterminate'
	
df = pd.read_csv("./result/Section5/ACMG.actionable.genes.GRCh38.vcf",  sep='\t')
df = df.copy()
df['label'] = df['pred_prob'].apply(classify)

# 2. Retain only pathogenic and benign groups
df = df[
	df['label'].isin([
		'pathogenic',
		'benign'
	])
].copy()

# 3. SpliceAI columns
splice_cols = [
	'SpliceAI_acc_gain',
	'SpliceAI_acc_loss',
	'SpliceAI_don_gain',
	'SpliceAI_don_loss'
]
# Convert to numeric values
for col in splice_cols:
	df[col] = pd.to_numeric(
		df[col],
		errors='coerce'
	)
	
# 4. gene-level analysis
all_results = []
for col in splice_cols:
	# Analyze each gene separately
	for gene, g in df.groupby("gene"):
		benign = g.loc[
			g["label"] == "benign",
			col
		].dropna()
		pathogenic = g.loc[
			g["label"] == "pathogenic",
			col
		].dropna()
		# Require at least one sample
		if (
			len(benign) < 5 or
			len(pathogenic) < 5
		):
			continue
		# Mann-Whitney U
		stat, p = mannwhitneyu(
			benign,
			pathogenic,
			alternative='two-sided'
		)
		n1 = len(benign)
		n2 = len(pathogenic)

		# RBC effect size
		# rbc = 1 - (2 * stat) / (n1 * n2)
		# cliff's delta
		cd = cliffs_delta(pathogenic, benign)

		all_results.append({
			"feature": col,
			"gene": gene,
			"pvalue": p,
			"U": stat,
			# "RBC": rbc,
			"CD": cd,
			"n_benign": n1,
			"n_pathogenic": n2
		})

results_df = pd.DataFrame(all_results)

# 5. FDR correction
results_df["fdr"] = np.nan
for col in splice_cols:
	mask = results_df["feature"] == col
	results_df.loc[mask, "fdr"] = multipletests(
		results_df.loc[mask, "pvalue"],
		method='fdr_bh'
	)[1]

### D) Gene-level plot

In [ ]:
from adjustText import adjust_text
import matplotlib.pyplot as plt

# 6. volcano-like plots
mapping = {
	'SpliceAI_acc_gain': 'Acceptor gain',
	'SpliceAI_acc_loss': 'Acceptor loss',
	'SpliceAI_don_gain': 'Donor gain',
	'SpliceAI_don_loss': 'Donor loss'
}

for col in splice_cols:
	sub = results_df[
		results_df["feature"] == col
	].copy()
	sub["neglog10fdr"] = -np.log10(
		sub["fdr"] + 1e-300
	)
	plt.figure(figsize=(4.5, 4))

	# scatter
	plt.scatter(
		# sub["RBC"],
		sub["CD"],
		sub["neglog10fdr"],
		alpha=0.4,
		color="#103057"
	)

	# significance threshold
	plt.axhline(
		-np.log10(0.05),
		linestyle='--',
		linewidth=0.8,
		color="#666"
	)

	# effect direction baseline
	plt.axvline(
		0,
		linestyle='--',
		linewidth=0.8,
		color="#666"
	)

	# Annotate top genes
	extreme = (
		sub
		.query("fdr < 0.05")
		.sort_values(
			"neglog10fdr",
			ascending=False
		)
		.head(5)
	)

	texts = []
	for _, r in extreme.iterrows():
		texts.append(plt.text(
			# r["RBC"] + 0.005,
			r["CD"] + 0.005,
			r["neglog10fdr"],
			r["gene"],
			fontsize=10,
			fontstyle='italic'
		))
	adjust_text(
		texts,
		arrowprops=dict(arrowstyle='-', color='gray', lw=1),
		expand_text=(1.2, 1.2),
		expand_points=(1.2, 1.2)
	)
	# axis
	# plt.xlabel("RBC", fontsize=11)
	plt.xlabel("Effect size", fontsize=11)
	plt.ylabel("-log10(FDR)",  fontsize=11)
	plt.title(mapping[col], fontsize=11)
	plt.tick_params(axis='both', labelsize=10)
	plt.ylim(-3, sub["neglog10fdr"].max()*1.1)

	# border
	ax = plt.gca()
	for spine in ax.spines.values():
		spine.set_linewidth(.5)
		spine.set_color('#333')
	plt.tight_layout()
	plt.show()